# Pembersihan Korpus PDF Putusan PN Surabaya

Tujuan: mengubah 273 PDF di `PDF_Putusan/` menjadi teks bersih di `CLEAN_TXT/`
plus satu file CSV ringkas (`_MANIFEST.csv`) yang berisi nomor perkara dan nama hakim.

**Cara pakai:** jalankan cell berurutan dari atas ke bawah (Kernel > Restart & Run All).
Ubah `DRY_RUN = True` di cell **Konfigurasi** (cell kode pertama) kalau hanya ingin melihat rencana tanpa memindah file.

**Prasyarat:** `poppler-utils` terinstal (`pdftohtml`, `pdfinfo`).
Hanya memakai library standar Python (`re`, `shutil`, `hashlib`, `csv`, `subprocess`).

**Struktur notebook:** konfigurasi & pola regex di bagian atas, lalu beberapa fungsi utama per tahap, lalu satu cell untuk menjalankan tiap tahap.

In [1]:
# ================= KONFIGURASI & IMPORT (satu-satunya bagian yang perlu diubah) ==
import csv
import hashlib
import html
import re
import shutil
import subprocess
from collections import Counter
from pathlib import Path

# ---------- Konfigurasi (satu-satunya bagian yang perlu diubah) ----------

FOLDER_SUMBER = Path("PDF_Putusan")
FOLDER_BERSIH  = Path("CLEAN_TXT")
FOLDER_SISIH   = Path("_EXCLUDE")
FILE_MANIFEST  = Path("_MANIFEST.csv")
DRY_RUN   = False   # True = hanya mencetak rencana, tidak memindah/menulis apa pun
BATAS_FILE = 0      # 0 = proses semua file; isi angka kecil (mis. 5) untuk uji coba

# ---------- Konstanta filter font (hasil audit: berlaku di 273/273 file) ---
UKURAN_WATERMARK  = 40        # watermark "Mahkamah Agung Republik Indonesia" selalu >= 40pt
WARNA_HEADER      = "#3b3b3b"  # header "Direktori Putusan" + URL selalu warna ini
UKURAN_DISCLAIMER = 8         # blok Disclaimer selalu 8pt
TOLERANSI_BARIS   = 4         # dua kotak teks dianggap sebaris bila beda 'top' <= 4 px
BATAS_TANPA_TEKS  = 2000      # file 1 halaman dengan teks < ini dianggap hasil scan
BATAS_ABORT       = 20        # pengaman: batalkan pemindahan bila file bermasalah > ini

KOLOM_MANIFEST = ["file_pdf", "nama_asli", "status", "alasan", "nomor_header",
         "nomor_footer", "nomor_cocok", "tahun", "jenis", "pengadilan",
         "hakim_ketua", "hakim_anggota", "hakim_tunggal", "cara_hakim",
         "halaman_pdf", "jumlah_baris", "jumlah_karakter"]

# ---------- Konfigurasi tahap 2: ekstraksi entitas ----------
INPUT_TXT = Path("CLEAN_TXT")
OUTPUT_CLEAN = Path("OUTPUT_CLEAN")
KOLOM_ENTITAS = ["nomor_putusan","nama_terdakwa_1","nama_terdakwa_2","nama_terdakwa_3","tuntutan_pidana","tuntutan_kuhp","tuntutan_hukuman","putusan_pidana","putusan_hukuman","tanggal_putusan","hakim_ketua","hakim_anggota_1","hakim_anggota_2","panitera","penuntut_umum"]
KOLOM_TAMBAH = ["file_txt","jenis_dokumen","entitas_berlaku","total_entitas_terisi","status_lengkap","catatan_kosong"]
KOLOM_CSV = KOLOM_ENTITAS + KOLOM_TAMBAH
TANDA_KOSONG = "-"
BATAS_PREVIEW = 5
CONTOH_DEMO = "T-1016_Pid.B_2026_PN Sby.txt"
LABEL_CETAK = {"nomor_putusan":"Nomor Putusan","nama_terdakwa_1":"Nama Terdakwa 1","nama_terdakwa_2":"Nama Terdakwa 2","nama_terdakwa_3":"Nama Terdakwa 3","tuntutan_pidana":"Tuntutan Pidana","tuntutan_kuhp":"Tuntutan KUHP","tuntutan_hukuman":"Tuntutan Hukuman","putusan_pidana":"Putusan Pidana","putusan_hukuman":"Putusan Hukuman","tanggal_putusan":"Tanggal Putusan","hakim_ketua":"Hakim Ketua","hakim_anggota_1":"Hakim Anggota 1","hakim_anggota_2":"Hakim Anggota 2","panitera":"Panitera","penuntut_umum":"Penuntut Umum"}
JEJAK_ATURAN = {"nomor_putusan":"baris judul 'Nomor :' + POLA_NOMOR","nama_terdakwa_1":"label Nama di blok identitas pidana","nama_terdakwa_2":"label Nama kedua di blok identitas pidana","nama_terdakwa_3":"label Nama ketiga di blok identitas pidana","tuntutan_pidana":"amar 1 Tuntutan + frasa sesudah 'bersalah melakukan'","tuntutan_kuhp":"amar 1 Tuntutan + pola 'Pasal ... KUHP'","tuntutan_hukuman":"amar 2 Tuntutan + durasi sesudah 'selama'","putusan_pidana":"amar 1 MENGADILI + frasa sesudah 'bersalah melakukan'","putusan_hukuman":"amar 2 MENGADILI + durasi sesudah 'selama'","tanggal_putusan":"kalimat 'Demikian ...' + pola 'pada hari, tanggal'","hakim_ketua":"nama sebelum 'sebagai/selaku Hakim Ketua'","hakim_anggota_1":"nama pertama sebelum 'masing-masing ... Anggota'","hakim_anggota_2":"nama kedua sebelum 'masing-masing ... Anggota'","panitera":"nama sesudah 'dibantu oleh' sebelum 'Panitera Pengganti'","penuntut_umum":"nama sesudah 'dihadiri oleh' sebelum 'Penuntut Umum'"}

# Alasan standar bila sebuah entitas kosong (dipakai ekstrak())
PESAN_KOSONG = {
    "nomor_putusan": "nomor tidak terbaca pada judul dokumen",
    "nama_terdakwa_1": "label Nama lengkap/terdakwa tidak ditemukan di blok identitas",
    "nama_terdakwa_2": "dokumen ini tidak mempunyai terdakwa kedua/ketiga",
    "nama_terdakwa_3": "dokumen ini tidak mempunyai terdakwa kedua/ketiga",
    "tuntutan_pidana": "amar Tuntutan nomor 1 tidak memuat perbuatan yang dapat dipisahkan",
    "tuntutan_kuhp": "amar Tuntutan nomor 1 tidak memuat Pasal KUHP",
    "tuntutan_hukuman": "amar Tuntutan nomor 2 tidak memuat durasi sesudah 'selama'",
    "putusan_pidana": "amar MENGADILI nomor 1 tidak memuat perbuatan yang dapat dipisahkan",
    "putusan_hukuman": "amar MENGADILI nomor 2 tidak memuat durasi sesudah 'selama'",
    "tanggal_putusan": "tanggal 'pada hari ..., tanggal ...' tidak ditemukan di penutup",
    "hakim_ketua": "nama sebelum 'Hakim Ketua' tidak ditemukan di penutup",
    "hakim_anggota_1": "daftar 'masing-masing sebagai Hakim Anggota' tidak ditemukan (misalnya hakim tunggal)",
    "hakim_anggota_2": "anggota kedua tidak ditemukan (misalnya hakim tunggal)",
    "panitera": "pola 'dibantu oleh ... Panitera Pengganti' tidak ditemukan",
    "penuntut_umum": "nama sebelum 'Penuntut Umum' tidak ditemukan di penutup",
}

In [2]:
# ================= TAHAP 1 - POLA REGEX & KAMUS =================================
# Keputusan penyaringan berdasar UKURAN dan WARNA font, bukan menebak isi teks.

# Kamus perbaikan kata berhuruf renggang ("P U T U S A N" -> "PUTUSAN").
# Hanya berisi kata yang sudah diverifikasi muncul di korpus ini.
KATA_RENGGANG = {
    "P U T U S A N": "PUTUSAN", "P E N E T A P A N": "PENETAPAN",
    "M E N G A D I L I": "MENGADILI", "M E N E T A P K A N": "MENETAPKAN",
    "M E N U N T U T": "MENUNTUT", "M E L A W A N": "MELAWAN",
    "S U B S I D A I R": "SUBSIDAIR", "K E D U A": "KEDUA",
    "A T A U": "ATAU", "A t a u": "Atau", "M A K A": "MAKA",
    "M a k a": "Maka", "B a h w a": "Bahwa", "N a m a": "Nama",
    "P e k e r j a a n": "Pekerjaan", "A g a m a": "Agama",
    "L a w a n": "Lawan", "L A W A N": "LAWAN", "l a w a n": "lawan",
    "M e l a w a n": "Melawan", "U m u r": "Umur",
    "J u m l a h": "Jumlah", "m e m b a c a": "membaca",
    "m e n d e n g a r": "mendengar", "M e i": "Mei", "J u l i": "Juli",
    "N I H I L": "NIHIL", "A T K": "ATK", "S T M": "STM", "S M K": "SMK",
    "L A W F I R M": "LAW FIRM", "R O C H M A": "ROCHMA",
    "Y U N A W A T": "YUNAWAT", "G U M I N A": "GUMINA",
}

POLA_BARIS_SAMPAH = re.compile(r"^[-\u2013\u2014:.\s]+$")

# --- pola nama hakim ---
GELAR = (r"(?:S\.?H\.?|SH\.?|M\.?H\.?|M\.?Hum\.?|Dr\.?|Dra\.?|Drs\.?"
         r"|H\.?J\.?|CPArb\.?|S\.?E\.?|SE\.?|MM\.?|H\.?S\.?|M\.?K\.?)")
POLA_NAMA = re.compile(rf"[A-Z][\w.\'-]*(?:\s+[A-Z][\w.\'-]*)*[,.]?\s*{GELAR}(?:\s*[,.]\s*{GELAR})*")
POLA_KELEBIH = re.compile(r"(?i)diputuskan dalam|ditetapkan dalam|"
                           r"rapat permusyawaratan|oleh\s+kami")
POLA_ANCHOR = re.compile(r"(?i)panitera\s+pengganti\s*[,;]?\s*hakim"
                         r"|hakim\s+tunggal"
                         r"|hakim\s*(?:-?\s*hakim\s*)?(?:ketua|anggota)")
POLA_SISA = re.compile(r"(?i)^(halaman|hal)\b|\bdari\s+\d+\s+(hal|halaman)"
                        r"|^(t\s*\.?\s*t\s*\.?\s*d)")
POLA_TTD = re.compile(r"^(t\s*\.?\s*t\s*\.?\s*d\s*\.?\s*)+$", re.I)
POLA_TUNGGAL_KALIMAT = re.compile(
    r"(?i)(.{0,80}?)\s*,?\s*(?:selaku\s+)?(?:hakim\s+pengadilan\s+negeri"
    r"|hakim\s+ketua\s+majelis)")

# --- pola nomor perkara & nama file ---
POLA_NOMOR = re.compile(r"(\d+)\s*/?\s*([A-Za-z][\w.\-]*(?:[ /][A-Za-z][\w.\-]*)?)\s*/\s*(\d{4})\s*/\s*P\.?\s*N\.?\s*([\w.\s]*)")
POLA_NAMA_FILE = re.compile(r"^(\d+)_([\w.\-]+?)_(\d{4})_PN", re.I)

# --- validasi ---
TRIGGER = {
    "nama (lengkap|terdakwa) :": r"nama (lengkap|terdakwa) :",
    "menyatakan terdakwa": r"menyatakan terdakwa",
    "menjatuhkan pidana": r"menjatuhkan pidana",
    "diputuskan dalam": r"diputuskan dalam",
    "hakim ketua": r"hakim ketua",
}

NOISE = [r"^Mahkamah Agung Republik Indonesia$", r"mahkamahagung\.go\.id",
         r"^Disclaimer$", r"^Direktori Putusan Mahkamah Agung Republik Indonesia$"]

## Tahap 1 — Fungsi utama pembersihan PDF

In [3]:
# ================= TAHAP 1 - FUNGSI UTAMA: baca & bersihkan satu PDF ===========
# Dua fungsi utama:
#   proses_satu_pdf(path_pdf) -> (daftar_baris_bersih, hakim)
#   ekstrak_hakim(daftar_baris) -> {"ketua", "anggota", "tunggal", "cara"}


def proses_satu_pdf(path_pdf):
    """Pipeline lengkap satu file:
    baca PDF -> saring watermark/header/disclaimer -> gabung jadi baris
    -> rapikan teks -> ambil nama hakim."""

    # --- 1. Baca PDF jadi kotak-kotak teks (poppler; -i = jangan tulis gambar) ---
    xmlek = subprocess.run(
        ["pdftohtml", "-xml", "-stdout", "-hidden", "-i", str(path_pdf)],
        capture_output=True, text=True,
    ).stdout

    # {id_font: (ukuran, keluarga, warna)}
    fontspec = {}
    pola_font = r'<fontspec id="(\d+)" size="([\d.]+)" family="([^"]+)" color="([^"]+)"'
    for cocok in re.finditer(pola_font, xmlek):
        fontspec[cocok.group(1)] = (float(cocok.group(2)), cocok.group(3), cocok.group(4))

    # daftar kotak: {halaman, atas, kiri, id_font, teks}
    kotak = []
    for nomor, halaman in enumerate(re.findall(r"<page.*?</page>", xmlek, re.S)):
        for cocok in re.finditer(r"<text ([^>]*)>(.*?)</text>", halaman, re.S):
            atribut = dict(re.findall(r'(\w+)="([^"]*)"', cocok.group(1)))
            teks = html.unescape(re.sub(r"<[^>]+>", "", cocok.group(2)))
            teks = re.sub(r"\s{2,}", " ", teks).strip()
            if teks:
                kotak.append({"halaman": nomor, "atas": int(atribut["top"]),
                              "kiri": int(atribut["left"]),
                              "id_font": atribut.get("font"), "teks": teks})

    # --- 2. Buang watermark / header situs / disclaimer (berdasar UKURAN & WARNA font) ---
    tersaring = []
    for k in kotak:
        ukuran, _, warna = fontspec.get(k["id_font"], (0, "", ""))
        if ukuran >= UKURAN_WATERMARK:
            continue
        if warna == WARNA_HEADER:
            continue
        if ukuran == UKURAN_DISCLAIMER:
            continue
        tersaring.append(k)

    # --- 3. Susun kotak menjadi baris (kotak yang tingginya mirip digabung sebaris) ---
    urut = sorted(tersaring, key=lambda k: (k["halaman"], k["atas"], k["kiri"]))
    gabungan, tampal = [], None
    for k in urut:
        kunci = (k["halaman"], round(k["atas"] / TOLERANSI_BARIS))
        if tampal and kunci == tampal["kunci"]:
            tampal["teks"] += " " + k["teks"]
        else:
            tampal = {"kunci": kunci, "teks": k["teks"]}
            gabungan.append(tampal)
    baris_mentah = [b["teks"] for b in gabungan]

    # --- 4. Rapikan: spasi, 'H a k i m', huruf renggang, baris tanda baca saja ---
    rapi = []
    for baris in baris_mentah:
        baris = re.sub(r"\s{2,}", " ", baris).strip()
        baris = re.sub(r"H\s+a\s+k\s+i\s+m", "Hakim", baris, flags=re.I)
        for renggang, rapat in KATA_RENGGANG.items():
            baris = baris.replace(renggang, rapat)
        if baris and not POLA_BARIS_SAMPAH.match(baris):
            rapi.append(baris)

    # --- 5. Sambung baris berakhiran '-' dengan baris berikut (tanda hubung SELALU
    #        dipertahankan: 'Laki-laki', 'masing-masing', 'perundang-undangan') ---
    tersambung = []
    for baris in rapi:
        if tersambung and tersambung[-1].endswith("-") and baris[:1].islower():
            tersambung[-1] = tersambung[-1] + baris
        else:
            tersambung.append(baris)

    # --- 6. Seragamkan label nomor: 'No.', 'NOMOR :', 'Perkara Nomor :' -> 'Nomor :' ---
    baris_akhir = []
    for baris in tersambung:
        baris = re.sub(r"(?i)^(?:perkara\s+)?n(?:omor|o)\.?\s*:?", "Nomor :", baris, count=1)
        baris_akhir.append(re.sub(r"^Nomor :(\S)", r"Nomor : \1", baris))

    return baris_akhir, ekstrak_hakim(baris_akhir)


def ekstrak_hakim(daftar_baris):
    """Ambil nama hakim. Tiga aturan dicoba berurutan; hasil pertama yang berhasil dipakai:
    1) 'oleh kami, X sebagai Hakim Ketua ...'
    2) blok tanda tangan 'Panitera Pengganti, Hakim, ...' + baris nama di bawahnya
    3) cadangan: 'X, S.H., Hakim Pengadilan Negeri Surabaya'"""

    def aturan_oleh_kami():
        for i, baris in enumerate(daftar_baris):
            if not POLA_KELEBIH.search(baris):
                continue
            kalimat = " ".join(daftar_baris[i:i + 7])
            cocok = re.search(r"(?i)(?:oleh\s+kami\s*[:,]?|oleh,?)\s*(.*?)\s*"
                              r"(?:masing-?\s*masing\s+)?sebagai\s+hakim\s+ketua",
                              kalimat)
            if cocok:
                ketua = (POLA_NAMA.findall(cocok.group(1)) or [None])[0]
                sisa = kalimat[cocok.end():]
                if "masing-masing sebagai" in sisa:
                    sisa = sisa.split("masing-masing sebagai")[0]
                anggota = POLA_NAMA.findall(sisa)
                if ketua or anggota:
                    return {"ketua": ketua, "anggota": anggota,
                            "tunggal": None, "cara": "oleh-kami"}
            if re.search(r"(?i)hakim\s+tunggal", kalimat):
                nama = POLA_NAMA.findall(kalimat)
                if nama:
                    return {"ketua": None, "anggota": [],
                            "tunggal": nama[0], "cara": "oleh-kami-tunggal"}
        return None

    def aturan_blok_ttd():
        for i, baris in enumerate(daftar_baris):
            if not POLA_ANCHOR.search(baris):
                continue
            tunggal = bool(re.search(r"(?i)hakim\s+tunggal", baris))
            for berikut in daftar_baris[i + 1:i + 6]:
                if POLA_SISA.search(berikut) or POLA_TTD.match(berikut.strip()):
                    continue
                nama = POLA_NAMA.findall(berikut)
                if not nama:
                    continue
                kecil = baris.lower()
                if tunggal:
                    return {"ketua": None, "anggota": [],
                            "tunggal": nama[0], "cara": "ttd-tunggal"}
                if "ketua" in kecil and "anggota" in kecil:
                    if re.search(r"(?i)anggota.*ketua", kecil):
                        ketua = nama[1] if len(nama) > 1 else None
                        return {"ketua": ketua, "anggota": [nama[0]],
                                "tunggal": None, "cara": "ttd-majelis"}
                    return {"ketua": nama[0], "anggota": nama[1:],
                            "tunggal": None, "cara": "ttd-majelis"}
                if "panitera" in kecil and "anggota" not in kecil:
                    return {"ketua": None, "anggota": [],
                            "tunggal": nama[-1], "cara": "ttd-tunggal"}
                if len(nama) >= 2:
                    return {"ketua": nama[0], "anggota": nama[1:],
                            "tunggal": None, "cara": "ttd-majelis"}
                return {"ketua": None, "anggota": [],
                        "tunggal": nama[0], "cara": "ttd-tunggal"}
        return None

    def aturan_hakim_kalimat():
        for i, baris in enumerate(daftar_baris):
            kalimat = " ".join(daftar_baris[i:i + 7])
            if POLA_KELEBIH.search(kalimat):
                continue  # sudah ditangani aturan_oleh_kami
            cocok = POLA_TUNGGAL_KALIMAT.search(kalimat)
            if cocok:
                nama = POLA_NAMA.findall(cocok.group(1))
                if nama:
                    return {"ketua": None, "anggota": [],
                            "tunggal": nama[-1], "cara": "tunggal-kalimat"}
        return None

    def bersihkan_nama(nama):
        """Buang sisa gelar di depan nama (mis. "M.H Teguh" -> "Teguh")."""
        for _ in range(2):
            baru = re.sub(r"^(?:S\.?H\.?|M\.?H\.?|M\.?Hum\.?|Drs?\.?|Dr\.?|H\.?J\.?)\s+",
                          "", nama).strip()
            if baru == nama:
                break
            nama = baru
        return nama

    for aturan in (aturan_oleh_kami, aturan_blok_ttd, aturan_hakim_kalimat):
        hasil = aturan()
        if hasil:
            if hasil["ketua"]:
                hasil["ketua"] = bersihkan_nama(hasil["ketua"])
            if hasil["tunggal"]:
                hasil["tunggal"] = bersihkan_nama(hasil["tunggal"])
            hasil["anggota"] = [bersihkan_nama(n) for n in hasil["anggota"]]
            return hasil
    return {"ketua": None, "anggota": [], "tunggal": None, "cara": "tidak-ditemukan"}

In [4]:
# ================= TAHAP 1 - FUNGSI UTAMA: audit korpus & rencana aksi ==========
#   audit_korpus()       -> {nama_pdf: info}
#   susun_rencana(info)  -> (rencana, nomor)


def audit_korpus():
    """Proses semua PDF (sumber + yang sudah disisihkan), kembalikan {nama: info}.
    Menyertakan _EXCLUDE/ supaya manifest tetap 273 baris saat notebook dijalankan ulang."""
    daftar = sorted(p.name for p in FOLDER_SUMBER.glob("*.pdf"))
    daftar += sorted(p.name for p in FOLDER_SISIH.glob("*.pdf"))
    if BATAS_FILE:
        daftar = daftar[:BATAS_FILE]

    info = {}
    for urutan, nama in enumerate(daftar, 1):
        lokasi = FOLDER_SUMBER / nama
        if not lokasi.exists():
            lokasi = FOLDER_SISIH / nama

        baris, hakim = proses_satu_pdf(lokasi)
        teks = "\n".join(baris)

        # jumlah halaman lewat pdfinfo (cepat, tanpa mengekstrak teks)
        keluar = subprocess.run(["pdfinfo", str(lokasi)],
                                capture_output=True, text=True).stdout
        cocok = re.search(r"Pages:\s+(\d+)", keluar)

        info[nama] = {
            "md5": hashlib.md5(Path(lokasi).read_bytes()).hexdigest(),  # deteksi duplikat byte-identik
            "halaman": int(cocok.group(1)) if cocok else 0,
            "baris": baris,
            "karakter": len(teks),
            "hakim": hakim,
        }
        if urutan % 25 == 0:
            print(f"  ... {urutan}/{len(daftar)} file")
    return info


def susun_rencana(info):
    """Tentukan aksi tiap file berdasar ISI dokumen (bukan daftar hafalan).

    Return:
      rencana : {nama: (aksi, alasan, nama_baru)}, aksi = 'normal' | 'sisihkan' | 'rename'
      nomor   : {nama: (nomor_header, nomor_footer, cocok)} -> dipakai untuk manifest
    """
    # peta md5 -> nama kanonis (nama yang bukan 'putusan_*' diutamakan)
    peta_md5 = {}
    for nama in sorted(info):
        md5 = info[nama]["md5"]
        if md5 not in peta_md5:
            peta_md5[md5] = nama
        elif (peta_md5[md5].lower().startswith("putusan_")
              and not nama.lower().startswith("putusan_")):
            peta_md5[md5] = nama

    # seragamkan untuk perbandingan (abaikan spasi/titik/besar-kecil)
    bersih = lambda s: re.sub(r"[^A-Z0-9]+", "", s.upper())

    rencana, nomor = {}, {}
    for nama in info:
        data = info[nama]
        baris = data["baris"]

        # --- nomor perkara resmi di 12 baris pertama (blok judul) ---
        # prioritas: baris berlabel 'Nomor :' (bukan 'Jo Nomor' / 'Register Nomor')
        header = None
        awal = baris[:12]
        for b in awal:
            if b.startswith("Nomor :"):
                cocok = POLA_NOMOR.search(b)
                if cocok:
                    header = (cocok.group(1), cocok.group(2), cocok.group(3), cocok.group(4))
                    break
        if header is None:
            for b in awal:
                cocok = POLA_NOMOR.search(b)
                if cocok:
                    header = (cocok.group(1), cocok.group(2), cocok.group(3), cocok.group(4))
                    break

        # --- 'Penetapan/Putusan Nomor ...' di footer halaman ---
        footer = None
        for b in baris[-15:]:
            if re.search(r"(?i)(penetapan|putusan).*nomor", b):
                cocok = POLA_NOMOR.search(b)
                if cocok:
                    footer = (cocok.group(1), cocok.group(2), cocok.group(3), cocok.group(4))
                    break

        # bandingkan nomor+jenis+tahun saja; ekor kalimat footer ("tertanggal ...") diabaikan
        cocok_nomor = ""
        if header and footer:
            h = (bersih(header[0]), bersih(header[1]), header[2])
            f = (bersih(footer[0]), bersih(footer[1]), footer[2])
            cocok_nomor = "ya" if h == f else "tidak"
        nomor[nama] = (header, footer, cocok_nomor)

        # --- tentukan aksi ---
        # '1031_Pdt.G_2026_PN Sby.pdf' -> ('1031', 'Pdt.G', '2026')
        cocok_file = POLA_NAMA_FILE.search(nama)
        pecah = cocok_file.groups() if cocok_file else None

        if peta_md5[data["md5"]] != nama:
            aksi = ("sisihkan", f"duplikat byte-identik dari {peta_md5[data['md5']]}", None)
        elif data["halaman"] <= 1 and data["karakter"] < BATAS_TANPA_TEKS:
            aksi = ("sisihkan", "hasil scan tanpa layer teks (butuh OCR)", None)
        elif pecah and header:
            nomor_f, jenis_f, tahun_f = bersih(pecah[0]), bersih(pecah[1]), pecah[2]
            nomor_h, jenis_h, tahun_h, adil_h = (bersih(header[0]), bersih(header[1]),
                                                 header[2], bersih(header[3]))
            if nomor_f != nomor_h:
                aksi = ("sisihkan", f"nomor nama file ({nomor_f}) != nomor isi ({nomor_h})", None)
            elif "SBY" not in adil_h and adil_h:
                aksi = ("sisihkan", f"putusan pengadilan lain ({adil_h})", None)
            elif tahun_f != tahun_h:
                baru = nama.replace(f"_{tahun_f}_", f"_{tahun_h}_", 1)
                aksi = ("rename", f"tahun nama file ({tahun_f}) -> tahun isi ({tahun_h})", baru)
            elif jenis_f != jenis_h:
                aksi = ("normal", f"flag: jenis nama file ({jenis_f}) != jenis isi ({jenis_h})", None)
            else:
                aksi = ("normal", "", None)
        elif pecah and not header:
            aksi = ("normal", "flag: nomor tidak terbaca di header", None)
        else:
            aksi = ("normal", "", None)
        rencana[nama] = aksi

    return rencana, nomor

In [5]:
# ================= TAHAP 1 - FUNGSI UTAMA: eksekusi, manifest, validasi =========
#   jalankan_aksi(info, rencana)         -> {nama: (nama_akhir, status, alasan)}
#   tulis_manifest(info, rencana, nomor) -> daftar baris manifest (dict)
#   validasi(info, rencana, data)        -> mencetak rekap
#   jalankan_pembersihan()               -> menjalankan seluruh tahap 1


def jalankan_aksi(info, rencana):
    """Eksekusi rencana (pindah ke _EXCLUDE, rename PDF, tulis TXT bersih).
    Idempoten. Berhenti bila file bermasalah melebihi BATAS_ABORT."""
    bermasalah = sum(1 for v in rencana.values() if v[0] != "normal")
    if bermasalah > BATAS_ABORT:
        raise SystemExit(f"BATAL: {bermasalah} file bermasalah melebihi batas {BATAS_ABORT}.")

    hasil = {}
    for nama in sorted(rencana):
        aksi, alasan, baru = rencana[nama]

        if aksi == "sisihkan":
            asal, tujuan = FOLDER_SUMBER / nama, FOLDER_SISIH / nama
            if tujuan.exists():
                status = "sudah-disana"
            elif DRY_RUN:
                status = "dry-run"
            else:
                FOLDER_SISIH.mkdir(exist_ok=True)
                shutil.move(str(asal), str(tujuan))
                status = "dipindah"
            hasil[nama] = ("exclude", status, alasan)
            continue

        # aksi 'rename' atau 'normal' -> keduanya menulis TXT bersih
        status_rename = None
        if aksi == "rename":
            asal, tujuan = FOLDER_SUMBER / nama, FOLDER_SUMBER / baru
            if not asal.exists():
                status_rename = "sudah-rename"
            elif tujuan.exists():
                status_rename = "batal-ada-kembar"
            elif DRY_RUN:
                status_rename = "dry-run"
            else:
                asal.rename(tujuan)
                status_rename = "direname"

        nama_txt = baru if aksi == "rename" else nama
        FOLDER_BERSIH.mkdir(exist_ok=True)
        tujuan_txt = FOLDER_BERSIH / ("T-" + Path(nama_txt).stem + ".txt")
        if not DRY_RUN:
            tujuan_txt.write_text("\n".join(info[nama]["baris"]) + "\n", encoding="utf-8")

        if aksi == "rename":
            hasil[nama] = (baru, f"{status_rename} + {tujuan_txt}", alasan)
        else:
            hasil[nama] = (nama, str(tujuan_txt), alasan)
    return hasil


def tulis_manifest(info, rencana, nomor):
    """Susun & tulis _MANIFEST.csv (aktif + rename + exclude) dan _EXCLUDE/_ALASAN.txt."""
    teks_nomor = lambda p: f"{p[0]}/{p[1]}/{p[2]}" if p else ""  # ('100','Pdt.G','2026') -> '100/Pdt.G/2026'

    baris_data = []
    for nama in sorted(info):
        data = info[nama]
        aksi, alasan, baru = rencana[nama]
        header, footer, cocok = nomor[nama]
        akhir = baru if aksi == "rename" else nama
        status = {"sisihkan": "exclude", "rename": "rename", "normal": "aktif"}[aksi]
        hakim = data["hakim"]

        if re.search(r"-{3,}.*S\\.?H", " ".join(data["baris"][-30:])):
            alasan = (alasan + "; " if alasan else "") + "flag: nama hakim tersensor"

        baris_data.append({
            "file_pdf": akhir, "nama_asli": nama, "status": status, "alasan": alasan,
            "nomor_header": teks_nomor(header), "nomor_footer": teks_nomor(footer),
            "nomor_cocok": cocok,
            "tahun": header[2] if header else "", "jenis": header[1] if header else "",
            "pengadilan": (header[3].strip() if header else ""),
            "hakim_ketua": hakim["ketua"] or "",
            "hakim_anggota": "; ".join(hakim["anggota"]),
            "hakim_tunggal": hakim["tunggal"] or "", "cara_hakim": hakim["cara"],
            "halaman_pdf": data["halaman"], "jumlah_baris": len(data["baris"]),
            "jumlah_karakter": data["karakter"],
        })

    if DRY_RUN:
        return baris_data
    with open(FILE_MANIFEST, "w", newline="", encoding="utf-8") as berkas:
        penulis = csv.DictWriter(berkas, fieldnames=KOLOM_MANIFEST)
        penulis.writeheader()
        penulis.writerows(baris_data)
    alasan_exclude = [f"{b['nama_asli']}: {b['alasan']}" for b in baris_data if b["status"] == "exclude"]
    (FOLDER_SISIH / "_ALASAN.txt").write_text("\n".join(alasan_exclude) + "\n", encoding="utf-8")
    return baris_data


def validasi(info, rencana, data):
    """Cetak rekap: file, karakter, sisa noise, cakupan trigger, file tanpa hakim, sampel teks."""
    aktif = [n for n, v in rencana.items() if v[0] != "sisihkan"]
    total_karakter = sum(info[n]["karakter"] for n in aktif)
    print(f"File aktif : {len(aktif)}")
    print(f"File sisihan: {len(info) - len(aktif)}")
    print(f"Total karakter bersih: {total_karakter:,}")

    print("\nSisa noise di CLEAN_TXT/ (harus 0 semua):")
    for pola in NOISE:
        kena = sum(sum(1 for b in info[n]["baris"] if re.search(pola, b, re.I)) for n in aktif)
        print(f"  {kena:6d}  {pola}")

    print("\nCakupan trigger rule (jumlah file yang cocok):")
    for nama_trigger, pola in TRIGGER.items():
        kena = sum(1 for n in aktif if any(re.search(pola, b, re.I) for b in info[n]["baris"]))
        print(f"  {kena:4d}/{len(aktif)}  {nama_trigger}")

    print("\nFile tanpa hakim:")
    for b in data:
        if b["status"] == "aktif" and not (b["hakim_ketua"] or b["hakim_anggota"] or b["hakim_tunggal"]):
            print(f"   {b['file_pdf']}")

    print("\nSampel 30 baris pertama (1 pidana, 1 gugatan, 1 permohonan):")
    for contoh in ["1002_Pid.Sus-LH_2026_PN Sby.pdf", "108_Pdt.G_2026_PN Sby.pdf",
                   "2296_Pdt.P_2026_PN Sby.pdf"]:
        if contoh in info and rencana[contoh][0] != "sisihkan":
            print(f"\n===== {contoh} =====")
            for baris in info[contoh]["baris"][:30]:
                print(" ", baris[:100])


def jalankan_pembersihan():
    """Jalankan seluruh tahap 1: audit -> rencana -> aksi -> manifest -> validasi."""
    print("Mulai audit korpus...")
    info = audit_korpus()
    print(f"Selesai: {len(info)} file diaudit.")

    rencana, nomor = susun_rencana(info)
    print(f"{'file':38s} {'aksi':10s} alasan")
    for nama in sorted(rencana):
        aksi, alasan, baru = rencana[nama]
        if aksi != "normal" or alasan:
            ekstra = f" -> {baru}" if baru else ""
            print(f"{nama:38s} {aksi:10s} {alasan}{ekstra}")
    n_bermasalah = sum(1 for v in rencana.values() if v[0] != "normal")
    print(f"\nFile bermasalah: {n_bermasalah} dari {len(rencana)}")

    hasil = jalankan_aksi(info, rencana)
    n_aktif = sum(1 for nama, (akhir, _, _) in hasil.items() if not str(akhir).startswith("exclude"))
    print(f"Selesai. File aktif: {n_aktif}, disisihkan: {len(hasil) - n_aktif}.")

    data = tulis_manifest(info, rencana, nomor)
    print(f"Manifest ditulis: {len(data)} baris -> {FILE_MANIFEST}")

    validasi(info, rencana, data)
    return info, rencana, hasil, data

### Jalankan tahap 1

In [6]:
# ================= JALANKAN TAHAP 1 ==========================================
CACHE, RENCANA, HASIL, DATA = jalankan_pembersihan()

Mulai audit korpus...
  ... 25/273 file
  ... 50/273 file
  ... 75/273 file
  ... 100/273 file
  ... 125/273 file
  ... 150/273 file
  ... 175/273 file
  ... 200/273 file
  ... 225/273 file
  ... 250/273 file
Selesai: 273 file diaudit.
file                                   aksi       alasan
1031_Pdt.G_2026_PN Sby.pdf             sisihkan   nomor nama file (1031) != nomor isi (100)
17_Pdt.Sus-Pailit-GLL_2025_PN.Niaga Sby.pdf normal     flag: jenis nama file (PDTSUSPAILITGLL) != jenis isi (PDTSUSGUGATANLAINLAIN)
3162_Pid.B_2017_PN SBY.pdf             sisihkan   nomor nama file (3162) != nomor isi (3150)
3749_Pid.B_2017_PN SBY.pdf             sisihkan   nomor nama file (3749) != nomor isi (113)
3882_Pid.Sus_2017_PN SBY.pdf           normal     flag: jenis nama file (PIDSUS) != jenis isi (PIDB)
527_Pid.Sus_2018_PN SBY.pdf            sisihkan   nomor nama file (527) != nomor isi (359)
748_Pid.C_2025_PN Sby.pdf              sisihkan   hasil scan tanpa layer teks (butuh OCR)
946_Pdt.G_2025_P

## Ringkasan hasil

Fungsi `validasi()` mencetak semuanya: jumlah file aktif vs disisihkan, sisa noise (harus nol),
cakupan tiap trigger rule, daftar file tanpa hakim, dan sampel teks untuk inspeksi manual.

File yang perlu perhatian manusia ada di kolom `alasan` pada `_MANIFEST.csv`
(mis. `flag: ...`). Khusus kolom hakim: cek 20 baris dengan `cara_hakim` =
`tunggal-kalimat` atau `ttd-tunggal` terhadap blok tanda tangan di PDF aslinya.

# BAGIAN 10 — EKSTRAKSI 15 ENTITAS DARI CLEAN_TXT
Tujuan: melanjutkan `clean_pdf.ipynb` menjadi satu file pipeline.
- Input: `CLEAN_TXT/` (258 file .txt, prefix `T-`)
- Output: `OUTPUT_CLEAN/` (per file `O-*.txt` berisi entitas terstruktur) + `OUTPUT_CLEAN/entitas_258.csv` (rangkuman 258 data)
- Desain: beberapa fungsi utama (rule-based), regex dikumpulkan di cell pola, komentar Bahasa Indonesia.
- Nilai kosong memakai `-`; kolom keterangan menjelaskan alasan setiap kekosongan.



In [7]:
# ================= TAHAP 2 - POLA REGEX & KONSTANTA =============================
POLA_BARIS_NOMOR = re.compile(r"(?i)^\s*(?:[IVX]+\.\s*)?(?:putusan\s+|penetapan\s+)?nomor\b")
POLA_NOMOR_PUTUSAN = re.compile(r"(?i)(\d{1,5}|-{5,})\s*/?\s*([A-Za-z][\w.\-]*(?:[ /][A-Za-z][\w.\-]*)*?)\s*/\s*(\d{4})(?:\s*/\s*P\s*\.?\s*N\s*\.?\s*((?:Niaga\s+)?Sby|SBY))?")
POLA_LABEL_NAMA = re.compile(r"(?i)(?:^|\b)(?:[IVX0-9]+\.\s*)?(?:nama\s+(?:lengkap|terdakwa)|namalengkap|nama)\s*:")
POLA_BATAS_IDENTITAS = re.compile(r"(?i)terdakwa\s+dita(?:han|ngkap)|telah\s+ditahan|terdakwa\s+berada|ditahan\s*/\s*ditangkap|dibantu\s+(?:oleh\s+)?(?:penasihat|penasehat|advokat|para)|pengadilan negeri tersebut")
POLA_LABEL_LAPANGAN = re.compile(r"(?i)^\s*(?:[IVX0-9]+\.\s*)?(?:tempat(?:\s+tanggal)?(?:\s+lahir)?|umur|jenis|kebangsaan|kewarganegaraan|agama|pekerjaan|pendidikan|alamat|tanggal\s+lahir)\b[^:\n]*:")
POLA_AWAL_TUNTUTAN = re.compile(r"(?i)tuntutan\s+(?:pidana|jaksa|penuntut)|pembacaan\s+tuntutan|tuntutan\s+dari|yang\s+pada\s+pokoknya\s+menuntut")
POLA_AKHIR_TUNTUTAN = re.compile(r"(?i)Setelah mendengar (?:pembelaan|permohonan|tanggapan)|mengajukan pembelaan|tanggapan .*tuntutan|Menimbang,\s*bahwa\s+Terdakwa\s+didakwa|Terdakwa diajukan.*dakwaan|dengan dakwaan sebagai berikut|^\s*PERTAMA\s*$")
POLA_MENGADILI = re.compile(r"(?i)^\s*(?:\d+\.\s*)?M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\s*:?\s*$")
POLA_PENUTUP = re.compile(r"(?i)Demikian(?:lah)?[,\s]+(?:(?:putusan|penetapan)(?: ini| tersebut| itu)?\s+)?(?:diambil|diputus(?:kan)?|ditetapkan|diucapkan)\b")
POLA_NOMOR_AMAR = re.compile(r"^\s*(\d+)\s*[.)]\s*(.*)$")
POLA_GARIS_AMAR = re.compile(r"(?i)^\s*[-•]\s+(?=.*\b(menyatakan|menjatuhkan|menetapkan|menghukum|membebankan|memerintahkan)\b)")
POLA_BARIS_SAMPAH_AMAR = re.compile(r"(?i)^\s*(?:Halaman\b|Hal\.?\s*\d+|Putusan\s+No\b|Penetapan\s+No\b|Putusan Nomor\b|Penetapan Nomor\b|\d+)\s*$")
POLA_SAMPAH_HALAMAN = re.compile(r"(?i)\bHal(?:aman)?\.?\s*\d+|\b\d+\s*dari\s*\d+\s*hal\b")
FRAGMEN_NAMA = r"[A-ZÀ-Ž][\w.'’“”`\-]*"
TOKEN_GELAR = r"(?:[A-Z][\w]*\.(?:[A-Z\w.]*\.?)?|S\.?H\.?|M\.?H\.?|M\.?Hum\.?(?:um)?|MKn\.?(?:H)?|Drs?\.?|Dr\.?|SH|MH|S\.?E\.?|SE|S\.?Si\.?(?:Hum)?|S\.?Pd\.?|S\.?Kom\.?(?:Hum)?|S\.?Sos\.?(?:Hum)?|S\.?T\.?(?:Hum)?|A\.?Md\.?(?:Kep)?|H\.?J\.?(?:Hum)?|S\.Hut\.?)"
PEMISAH_GELAR = fr"(?:[\s,]*,[\s,]*|\s+(?={TOKEN_GELAR}\b))"
RANTAI_GELAR = fr"(?:(?:[\s,]*,[\s,]*|\s+(?={TOKEN_GELAR}\b)){TOKEN_GELAR})*"
TEKS_NAMA = fr"{FRAGMEN_NAMA}(?:\s+{FRAGMEN_NAMA})*?{PEMISAH_GELAR}{TOKEN_GELAR}{RANTAI_GELAR}"
POLA_NAMA_BERGELAR = re.compile(fr"\b({TEKS_NAMA})")
POLA_PANEL_HAKIM = re.compile(r"(?s)(?i:oleh\s+kami|oleh)\s*[,:;]?\s*" fr"(?P<ketua>{TEKS_NAMA})\s*,?\s*(?i:sebagai|selaku)\s*(?:(?i:Hakim)\s*(?i:Ketua)(?:\s+Majelis)?|(?i:Ketua)\s*(?i:Majelis)(?:\s+(?i:Hakim))?)" r"\s*,?\s*(?P<anggota>.+?)\s+(?i:masing)[\s-]*(?i:masing)\s+(?i:sebagai|selaku)\s+(?i:Hakim)(?:-(?i:Hakim))?\s+(?i:Anggota)" r"(?:\s+Majelis)?")
POLA_PANEL_HAKIM_ADHOC = re.compile(r"(?s)(?i:oleh\s+kami|oleh)\s*[,:;]?\s*" fr"(?P<ketua>{TEKS_NAMA})\s*,?\s*(?:(?i:sebagai|selaku)\s+)?(?:(?i:Hakim)\s*(?i:Ketua)(?:\s+Majelis)?|(?i:Ketua)\s*(?i:Majelis)(?:\s+(?i:Hakim))?)" r"\s*,?\s*(?P<anggota>.+?)\s+(?i:sebagai|selaku)\s+(?i:Hakim)(?:-(?i:Hakim))?\s+(?i:Anggota)")
POLA_PANEL_TANPA_LABEL = re.compile(r"(?s)(?i:oleh\s+kami|oleh)\s*[,:;]?\s*" fr"(?P<ketua>{TEKS_NAMA})\s*,?\s*(?:(?i:sebagai|selaku)\s+)?(?:(?i:Hakim)\s*(?i:Ketua)(?:\s+Majelis)?|(?i:Ketua)\s*(?i:Majelis)(?:\s+(?i:Hakim))?)" r"\s*,\s*(?P<anggota>.+?)\s*,\s*(?:(?i:penetapan)|(?i:putusan))\s+(?i:tersebut)")
POLA_PANITERA = re.compile(fr"(?s)(?i:didampingi|dibantu|dihadiri)(?:(?i:\s+oleh))?\s*:?\s*(?P<nama>{TEKS_NAMA})(?i:.*?Panitera\s+Pengganti)")
POLA_JPU = re.compile(fr"(?s)(?i:dihadiri|hadiri|dihadapan)(?:(?i:\s+pula))?(?:(?i:\s+oleh))?\s*:?\s*(?P<nama>{TEKS_NAMA})(?i:.*?(?:Jaksa\s+)?Penuntut\s+Umum)")
POLA_JPU_POLOS = re.compile(fr"(?s)(?i:dihadiri|hadiri|dihadapan)(?:(?i:\s+pula))?(?:(?i:\s+oleh))?\s*:?\s*(?P<nama>[A-ZÀ-Ž][\w.'’\-]*(?:\s+[A-ZÀ-Ža-z][\w.'’\-]*){{0,6}}?)\s*,?\s*(?:(?i:Jaksa)\s+)?(?i:Penuntut)\s+(?i:Umum)")
POLA_TANGGAL = re.compile(r"(?i)pada\s+(?:hari\s*:?\s*)?((?:[A-Za-z’]\s*){2,12}?)\s*,?\s*(?:tanggal\s*:?\s*|tangga\s*:?\s*|tangal\s*:?\s*)?(\d{1,2})\s+((?:januari|februari|pemberi|pebruari|maret|april|mei|juni|juli|agustus|september|sepetember|oktober|nopember|november|desember))\s+(\d{4})")
POLA_HUKUMAN_KHUSUS = re.compile(r"(?i)\bpidana mati\b|\bhukuman mati\b|\bpenjara seumur hidup\b|\bseumur hidup\b")
SATUAN_WAKTU = {"tahun", "bulan", "minggu", "hari"}
KATA_BERHENTI_LAMA = {"dikurangi", "dikurangkan", "dipotong", "potong", "dengan", "apabila", "yang", "untuk", "agar", "supaya", "serta", "denda", "pidana", "penjara", "tetap", "ditahan", "masa", "tahanan"}
POLA_PANITERA_APOSISI = re.compile(fr"(?s)(?:[,;:]|\btersebut\b)\s*(?P<nama>{TEKS_NAMA})[ \t]*,?[ \t]*Panitera\s+Pengganti")

PETA_JENIS = {"Pid.B":"pidana","Pid.Sus":"pidana","Pid.Sus-LH":"pidana","Pid.Sus-TPK":"pidana","Pid.Pra":"praperadilan","Pdt.P":"permohonan","Pdt.G":"gugatan","Pdt.G.S":"gugatan-sederhana","Pdt.Bth":"bantahan","Pdt.Sus-PHI":"phi","Pdt.Sus-HKI_Merek":"hki-merek","Pdt.Sus-HKI":"hki-merek","Pdt.Sus-PKPU":"pailit-pkpu","Pdt.Sus-Pailit-GLL":"pailit-pkpu","Pdt.Sus-Renvoi Prosedur":"renvoi"}
BERLAKU_PIDANA = {"nomor_putusan","nama_terdakwa_1","nama_terdakwa_2","nama_terdakwa_3","tuntutan_pidana","tuntutan_kuhp","tuntutan_hukuman","putusan_pidana","putusan_hukuman","tanggal_putusan","hakim_ketua","hakim_anggota_1","hakim_anggota_2","panitera","penuntut_umum"}
BERLAKU_UMUM = {"nomor_putusan","tanggal_putusan","hakim_ketua","hakim_anggota_1","hakim_anggota_2","panitera"}

PETA_BULAN = {"januari":"Januari","februari":"Februari","pemberi":"Februari","pebruari":"Februari","maret":"Maret","april":"April","mei":"Mei","juni":"Juni","juli":"Juli","agustus":"Agustus","september":"September","sepetember":"September","oktober":"Oktober","nopember":"November","november":"November","desember":"Desember"}

In [8]:
# ================= TAHAP 2 - ALAT BANTU (dipakai berulang oleh fungsi utama) =====
def rapikan_teks(teks):
    """Rapikan spasi; buang ; , : di ujung, tetapi pertahankan titik gelar."""
    if not teks:
        return None
    teks = re.sub(r"\s+", " ", teks).strip().lstrip(";:,. ").rstrip(";, :").strip()
    if len(teks) >= 2 and teks[0] in "\"'“‘" and teks[-1] in "\"'”’":
        teks = teks[1:-1].strip()  # buang tanda kutip pembungkus luar
    elif teks[:1] in "'‘":
        teks = teks[1:].strip()  # kutip pembuka tanpa penutup
    return teks or None


def rapikan_nama(nama):
    """Rapikan nama: buang koma ganda dan beri spasi sesudah koma."""
    nama = rapikan_teks(nama)
    if not nama:
        return None
    nama = re.sub(r"(?<=[a-zà-ž])\.,(?=[A-Z])", ",", nama)
    return re.sub(r",(?=[^\s\d])", ", ", nama)


def himpunan_berlaku(jenis):
    """Pidana memakai 15 entitas; selain itu hanya 6 entitas umum."""
    return set(BERLAKU_PIDANA) if jenis == "pidana" else set(BERLAKU_UMUM)


def buang_baris_sampah(daftar):
    """Buang baris kosong dan footer halaman ('Halaman 8', 'Hal. 26 dari 27 hal. Put. Nomor ...')."""
    hasil = []
    for x in daftar:
        if not x.strip():
            continue
        rendah = x.lower()
        footer = bool(POLA_BARIS_SAMPAH_AMAR.match(x) or POLA_SAMPAH_HALAMAN.search(x)
                      or ("halaman" in rendah and ("putusan" in rendah or "penetapan" in rendah
                                                   or re.match(r"^\s*hal", rendah))))
        if not footer:
            hasil.append(x)
    return hasil


def ambil_item_amar(bagian, nomor, frasa=None):
    """Ambil satu amar bernomor ('1.', '1)', '2. ...' di tengah baris, '- ...')."""
    nomor = str(nomor)
    kata_amar = r"(?:menyatakan|menjatuhkan|menetapkan|menghukum|membebankan|memerintahkan)"
    if frasa:
        kunci = frasa.lower()
        for i, x in enumerate(bagian):
            if kunci in x.lower():
                jangkau = range(max(0, i - 5), min(len(bagian), i + 6))
                tepat = [k for k in jangkau if re.match(rf"^\s*{nomor}[.)]\s*(.*)$", bagian[k]) or re.search(rf"(?:^|\s){nomor}[.)]\s*(?:$|{kata_amar})", bagian[k], re.I)]
                garis = [k for k in jangkau if POLA_GARIS_AMAR.match(bagian[k])]
                target = tepat + [k for k in garis if k not in tepat]
                if not target:
                    continue
                k = target[0]
                mulai = min(i, k)
                akhir = len(bagian)
                for j in range(max(i, k) + 1, len(bagian)):
                    if POLA_NOMOR_AMAR.match(bagian[j]):
                        akhir = j
                        break
                else:
                    for j in range(max(i, k) + 1, len(bagian)):
                        if POLA_GARIS_AMAR.match(bagian[j]):
                            akhir = j
                            break
                return " ".join(buang_baris_sampah(bagian[mulai:akhir]))
    for i, x in enumerate(bagian):
        cocok = POLA_NOMOR_AMAR.match(x)
        if cocok and cocok.group(1) == nomor:
            akhir = len(bagian)
            for j in range(i + 1, len(bagian)):
                if POLA_NOMOR_AMAR.match(bagian[j]) or POLA_GARIS_AMAR.match(bagian[j]):
                    akhir = j
                    break
            return " ".join(buang_baris_sampah(bagian[i:akhir]))
    return None


def ambil_perbuatan(teks):
    """Ambil nama perbuatan pidana sesudah 'bersalah melakukan' (dan varian-variannya)."""
    pola = re.search(r"(?i)bersalah\s+melakukan\s+(.*)", teks)
    if not pola:
        pola = re.search(r"(?i)\bmelakukan\b\s+(.*)", teks)
    if not pola:  # varian: '... menurut hukum dalam hal perbuatan X ...'
        pola = re.search(r"(?i)terbukti\b.{0,120}?menurut hukum\s+dalam hal\s+(?:perbuatan\s+)?(.*)", teks)
    if not pola:
        return None
    depan = re.split(r"(?i)\s*,?\s*sebagaimana|;|\bkarena\b|\bmelanggar\b|\bdiatur\b|\bdakwaan\b", pola.group(1), maxsplit=1)[0]
    depan = re.sub(r"^(?:[A-ZÀ-Ž][A-ZÀ-Ž.'’\-]*\s+){1,3}(?=[Tt]indak\s+[Pp]idana)", "", depan)
    depan = re.sub(r"(?i)^(?:perbuatan\s*,\s*|tindak\s+pidana\s+|pidana\s+|telah\s+melakukan\s+)+", "", depan)
    depan = rapikan_teks(depan)
    if depan and depan.lower() in {"tindak pidana", "pidana", "perbuatan", "tindak pidana tersebut"}:
        return None  # hanya label generik, bukan nama perbuatannya
    return depan


def ambil_lama_pidana(teks):
    """Ambil durasi penjara sesudah kata 'selama' ('1 tahun dan 6 bulan') atau hukuman khusus."""
    if not teks:
        return None
    khusus = POLA_HUKUMAN_KHUSUS.search(teks)
    if khusus:
        return rapikan_teks(khusus.group(0))
    for awal in re.finditer(r"(?i)\bs(?:elama|leama)\b\s*:?\s*(.{0,300})", teks):
        # rangkai token angka + satuan waktu sampai kata penghenti
        token = re.findall(r"\(|\)|,|;|:|\d+(?:[.,]\d+)?|[A-Za-zÀ-ž]+", awal.group(1))
        potongan = []
        ada_satuan = False
        i = 0
        while i < len(token):
            kata = token[i]
            rendah = kata.casefold()
            if kata in {",", ";", ":"}:
                break
            if rendah in KATA_BERHENTI_LAMA:
                break
            if rendah == "dan" or rendah == "clan":  # 'clan' = salah ketik OCR untuk 'dan'
                berikut = token[i + 1] if i + 1 < len(token) else ""
                if re.fullmatch(r"\d+(?:[.,]\d+)?", berikut):
                    potongan.append(kata)
                    i += 1
                    continue
                break
            potongan.append(kata)
            if rendah in SATUAN_WAKTU:
                ada_satuan = True
            i += 1
        if not ada_satuan:
            continue
        gabung = " ".join(potongan)
        gabung = re.sub(r"\(\s+", "(", gabung)
        gabung = re.sub(r"\s+\)", ")", gabung)
        hasil = rapikan_teks(gabung)
        if hasil:
            return hasil
    return None

In [9]:
# ================= TAHAP 2 - FUNGSI UTAMA: baca, petakan, ekstrak ===============
#   baca_dokumen(lintasan)                      -> baris, baris_low, jenis
#   petakan_bagian(baris, baris_low, jenis)     -> batas tiap bagian + teks penutup
#   ekstrak_identitas(baris, jenis, akhir_id)   -> nomor putusan + 3 terdakwa
#   ekstrak_amar(baris, bagian, jenis)          -> 3 tuntutan + 2 putusan
#   ekstrak_penutup(penutup, berlaku)           -> tanggal, hakim, panitera, penuntut umum


def baca_dokumen(lintasan):
    """Baca .txt -> (baris asli, baris huruf kecil, jenis dokumen dari nama file T-<nomor>_<jenis>_<tahun>...)."""
    try:
        teks = lintasan.read_text(encoding="utf-8", errors="replace")
    except Exception:
        teks = lintasan.read_text(encoding="latin-1", errors="replace")
    baris = teks.splitlines()
    baris_low = [x.lower() for x in baris]

    inti = Path(lintasan.name).stem
    if inti.startswith("T-"):
        inti = inti[2:]
    bagian_nama = inti.split("_", 2)
    token = bagian_nama[1] if len(bagian_nama) > 1 else ""
    return baris, baris_low, PETA_JENIS.get(token, "lainnya")


def petakan_bagian(baris, baris_low, jenis):
    """Tentukan batas tiap bagian dokumen.

    Return dict:
      identitas : indeks akhir blok identitas (hanya pidana)
      tuntutan  : (awal, akhir) mulai pembacaan tuntutan s.d. pembelaan/dakwaan (hanya pidana)
      putusan   : (awal, akhir) amar dari MENGADILI terakhir s.d. penutup (hanya pidana)
      penutup   : teks kalimat 'Demikian ...' yang sudah dibersihkan
    """
    bagian = {"identitas": None, "tuntutan": (None, None), "putusan": (None, None)}

    # --- penutup: kalimat 'Demikian ...' terakhir ---
    mulai = None
    for i, x in enumerate(baris):
        if POLA_PENUTUP.search(x):
            mulai = i
    if mulai is None:
        mulai = max(0, len(baris) - 80)
    awal_p, akhir_p = mulai, min(len(baris), mulai + 45)

    if jenis == "pidana":
        # --- identitas: di awal, sebelum status penahanan/pendampingan ---
        bagian["identitas"] = min(len(baris), 100)
        for i, x in enumerate(baris[:200]):
            if POLA_BATAS_IDENTITAS.search(x):
                bagian["identitas"] = i
                break

        # --- tuntutan: dari pembacaan tuntutan sampai sebelum pembelaan/dakwaan ---
        for i, x in enumerate(baris_low):
            if POLA_AWAL_TUNTUTAN.search(x):
                sekitar = " ".join(baris_low[max(0, i - 2):i + 8])
                if "penuntut umum" not in sekitar and "jaksa" not in sekitar and "menuntut" not in sekitar:
                    continue
                akhir_t = min(len(baris), i + 400)
                for j in range(i + 1, min(len(baris), i + 400)):
                    if (POLA_AKHIR_TUNTUTAN.search(baris[j])
                            or POLA_MENGADILI.match(baris[j])
                            or POLA_PENUTUP.search(baris[j])
                            or re.match(r"^\s*Menimbang", baris[j])):
                        akhir_t = j
                        break
                bagian["tuntutan"] = (i, akhir_t)
                break

        # --- putusan: MENGADILI terakhir sebelum penutup (termasuk huruf direnggangkan) ---
        penanda = [i for i, x in enumerate(baris) if POLA_MENGADILI.match(x)]
        if penanda:
            kandidat = [i for i in penanda if awal_p is None or i < awal_p]
            awal_u = kandidat[-1] if kandidat else penanda[-1]
            akhir_u = awal_p if awal_p is not None and awal_p > awal_u else min(len(baris), awal_u + 120)
            bagian["putusan"] = (awal_u, akhir_u)

    # --- teks penutup: gabung, buang footer, rapatkan titik gelar ---
    teks = " ".join(buang_baris_sampah(baris[awal_p:akhir_p]))
    teks = re.sub(r"\s+\.", ".", teks)  # 'M.Hum .' -> 'M.Hum.'
    teks = re.sub(r"\b([A-Z])\s+(?=[A-Z]\b)", r"\1", teks)  # 'S H' -> 'SH'
    teks = re.sub(r",\.(?=\s|$)", ",", teks)  # 'MH,.' -> 'MH,'
    teks = re.sub(r"\.(?=Hakim|Panitera|Penuntut|Jaksa)", ". ", teks)  # 'M.Hum.Hakim' -> 'M.Hum. Hakim'
    teks = re.sub(r"([A-Za-z])-([a-z])", r"\1\2", teks)  # 'tang-gal' -> 'tanggal'
    teks = re.sub(r"\bNomor\s*:\s*", "", teks)  # buang label 'Nomor :' yang terpotong footer
    bagian["penutup"] = teks
    return bagian


def ekstrak_identitas(baris, jenis, akhir_identitas):
    """Ambil nomor putusan (30 baris pertama) dan nama terdakwa 1-3 (hanya pidana)."""
    # --- nomor putusan: prioritas baris berlabel 'Nomor', lalu baris mana pun ---
    cocok = None
    for x in baris[:30]:
        if POLA_BARIS_NOMOR.search(x):
            kanan = x.split(":", 1)[1] if ":" in x else x
            cocok = POLA_NOMOR_PUTUSAN.search(kanan)
            if cocok:
                break
    if not cocok:
        for x in baris[:30]:
            cocok = POLA_NOMOR_PUTUSAN.search(x)
            if cocok:
                break

    nomor_putusan = None
    if cocok:
        angka = cocok.group(1).strip()
        if set(angka) != {"-"}:  # nomor tersensor semua '-----' dianggap tidak terbaca
            jenis_perkara = re.sub(r"\s*/\s*", "/", cocok.group(2).strip())
            tahun = cocok.group(3).strip()
            pengadilan = cocok.group(4)
            if pengadilan:
                pengadilan = re.sub(r"\s+", " ", pengadilan).rstrip(".")
                nomor_putusan = f"{angka}/{jenis_perkara}/{tahun}/PN {pengadilan}"
            else:
                nomor_putusan = f"{angka}/{jenis_perkara}/{tahun}"

    # --- nama terdakwa (maksimal 3) ---
    nama = []
    if jenis == "pidana":
        for i, x in enumerate(baris[:akhir_identitas]):
            if POLA_LABEL_NAMA.search(x):
                mentah = POLA_LABEL_NAMA.split(baris[i], maxsplit=1)[-1]
                calon = [mentah]
                if ";" not in mentah:  # nama bisa tersambung ke baris-baris berikutnya
                    for j in range(i + 1, min(akhir_identitas, i + 5)):
                        berikut = baris[j]
                        if not berikut.strip():
                            break
                        if POLA_LABEL_NAMA.search(berikut) or POLA_LABEL_LAPANGAN.search(berikut):
                            break
                        calon.append(berikut)
                        if ";" in berikut:
                            break
                gabung = " ".join(calon)
                bersih = None
                if gabung:  # ambil nama sebelum ';' dan buang keterangan lahir/alamat
                    depan = re.split(r";", gabung, maxsplit=1)[0]
                    depan = re.split(r",\s*(?:lahir|bertempat|selanjutnya|disebut|pekerjaan|alamat|jenis|agama|kebangsaan|umur|tempat|status|kewarganegaraan)\b", depan, maxsplit=1, flags=re.I)[0]
                    bersih = rapikan_teks(depan)
                if bersih and len(bersih) > 3 and bersih not in nama:
                    nama.append(bersih)
            if len(nama) >= 3:
                break
    nama = (nama + [None, None, None])[:3]

    return {"nomor_putusan": nomor_putusan, "nama_terdakwa_1": nama[0],
            "nama_terdakwa_2": nama[1], "nama_terdakwa_3": nama[2]}


def ekstrak_amar(baris, bagian, jenis):
    """Ambil tuntutan (perbuatan, pasal KUHP, lama hukuman) dan putusan (perbuatan, lama hukuman).
    Tuntutan/putusan: amar nomor 1 memuat perbuatan, amar nomor 2 memuat durasi 'selama'."""
    hasil = {"tuntutan_pidana": None, "tuntutan_kuhp": None, "tuntutan_hukuman": None,
             "putusan_pidana": None, "putusan_hukuman": None}
    if jenis != "pidana":
        return hasil

    awal, akhir = bagian["tuntutan"]
    if awal is not None:
        potongan = baris[awal:akhir]
        item1 = ambil_item_amar(potongan, "1", "menyatakan")
        if item1:
            hasil["tuntutan_pidana"] = ambil_perbuatan(item1)
            pasal = re.search(r"(?i)pasal\s+\d+.{0,200}?\s+KUH\s*Pidana|pasal\s+\d+.{0,200}?\s+KUHP", item1)
            hasil["tuntutan_kuhp"] = rapikan_teks(pasal.group(0)) if pasal else None
        item2 = ambil_item_amar(potongan, "2", "menjatuhkan")
        if item2:
            hasil["tuntutan_hukuman"] = ambil_lama_pidana(item2)

    awal, akhir = bagian["putusan"]
    if awal is not None:
        potongan = baris[awal:akhir]
        item1 = ambil_item_amar(potongan, "1", "menyatakan")
        if item1:
            hasil["putusan_pidana"] = ambil_perbuatan(item1)
        item2 = ambil_item_amar(potongan, "2", "menjatuhkan")
        if item2:
            hasil["putusan_hukuman"] = ambil_lama_pidana(item2)
    return hasil


def ekstrak_penutup(penutup, berlaku):
    """Ambil tanggal putusan, hakim ketua/anggota, panitera, penuntut umum dari kalimat penutup."""

    def terdekat(pola):
        """Pilih cocok dengan rentang terpendek (nama paling dekat dengan jabatannya)."""
        semua = list(pola.finditer(penutup))
        return min(semua, key=lambda m: m.end() - m.start()) if semua else None

    # --- tanggal putusan ---
    tanggal = None
    cocok = POLA_TANGGAL.search(penutup)
    if cocok:
        bulan = PETA_BULAN.get(cocok.group(3).lower(), cocok.group(3))
        tanggal = f"{int(cocok.group(2))} {bulan} {cocok.group(4)}"

    # --- panel hakim: pola baku, ad-hoc, lalu tanpa label anggota ---
    panel = (POLA_PANEL_HAKIM.search(penutup) or POLA_PANEL_HAKIM_ADHOC.search(penutup)
             or POLA_PANEL_TANPA_LABEL.search(penutup))

    ketua = None
    anggota = []
    if panel:
        ketua = rapikan_nama(panel.group("ketua"))
        for nama in POLA_NAMA_BERGELAR.findall(panel.group("anggota")):
            bersih = rapikan_nama(nama)
            if bersih and len(bersih) > 3 and bersih not in anggota:
                anggota.append(bersih)
    else:  # cadangan: 'oleh [kami] NAMA ... Hakim' tanpa label ketua/anggota
        for cocok in re.finditer(r"(?i:\boleh)(?:\s+kami)?\s*[,:;]?\s*(" + TEKS_NAMA + r")", penutup):
            sesudah = penutup[cocok.end():cocok.end() + 250]
            if re.search(r"(?i)\bHakim\b(?!\s+(?:Anggota|Ketua|Majelis|-|,))", sesudah) and "masing-masing" not in sesudah.lower():
                ketua = rapikan_nama(cocok.group(1))
                break

    # --- panitera pengganti (cadangan: aposisi ', NAMA, Panitera Pengganti') ---
    cocok = terdekat(POLA_PANITERA) or terdekat(POLA_PANITERA_APOSISI)
    panitera = rapikan_nama(cocok.group("nama")) if cocok else None

    # --- penuntut umum (hanya pidana; cadangan: nama tanpa gelar) ---
    penuntut_umum = None
    if "penuntut_umum" in berlaku:
        cocok = terdekat(POLA_JPU)
        if cocok:
            penuntut_umum = rapikan_teks(cocok.group("nama"))
        else:
            cocok = terdekat(POLA_JPU_POLOS)
            if cocok:
                nama = rapikan_nama(cocok.group("nama"))
                if nama and not re.search(r"(?i)\b(terdakwa|pemohon|advokat|kuasa|hakim|panitera|para|yang)\b", nama):
                    penuntut_umum = nama

    return {"tanggal_putusan": tanggal, "hakim_ketua": ketua,
            "hakim_anggota_1": anggota[0] if len(anggota) >= 1 else None,
            "hakim_anggota_2": anggota[1] if len(anggota) >= 2 else None,
            "panitera": panitera, "penuntut_umum": penuntut_umum}

In [10]:
# ================= TAHAP 2 - FUNGSI UTAMA: orkestrator & pengendali ==============
#   ekstrak(lintasan_txt)       -> 15 entitas + keterangan satu file
#   tulis_keluaran_txt(...)     -> tulis O-*.txt
#   demo_satu_file()            -> demo 1 file + jejak aturan
#   proses_batch()              -> semua file -> O-*.txt + entitas_258.csv
#   tampilkan_ringkasan()       -> 5 data teratas + rekap + cakupan tiap entitas


def ekstrak(lintasan_txt: Path):
    """Ekstrak 15 entitas dan keterangan dari satu file."""
    try:
        baris, baris_low, jenis = baca_dokumen(lintasan_txt)
        berlaku = himpunan_berlaku(jenis)
        bagian = petakan_bagian(baris, baris_low, jenis)

        # kumpulkan semua nilai yang ditemukan (None = tidak ditemukan)
        ditemukan = {}
        ditemukan.update(ekstrak_identitas(baris, jenis, bagian["identitas"]))
        ditemukan.update(ekstrak_amar(baris, bagian, jenis))
        ditemukan.update(ekstrak_penutup(bagian["penutup"], berlaku))

        # isi hasil; alasan kosong dicatat per entitas
        hasil = {kunci: TANDA_KOSONG for kunci in KOLOM_ENTITAS}
        alasan = {}
        for kunci in KOLOM_ENTITAS:
            if kunci not in berlaku:
                alasan[kunci] = f"tidak berlaku untuk dokumen {jenis}"
            elif ditemukan.get(kunci):
                hasil[kunci] = ditemukan[kunci]
            else:
                alasan[kunci] = PESAN_KOSONG[kunci]

        terisi = sum(1 for kunci in berlaku if hasil.get(kunci) not in (TANDA_KOSONG, None, ""))
        if terisi == len(berlaku):
            status = "lengkap"
        elif terisi == 0:
            status = "kosong"
        else:
            status = "sebagian"
        hilang = [kunci for kunci in KOLOM_ENTITAS if hasil.get(kunci) == TANDA_KOSONG]
        catatan = "; ".join(f"{kunci}: {alasan.get(kunci, 'tidak ditemukan')}" for kunci in hilang)
        return {**hasil, "_ok": True, "_error": None, "file_txt": lintasan_txt.name,
                "jenis_dokumen": jenis, "entitas_berlaku": len(berlaku),
                "total_entitas_terisi": sum(1 for kunci in KOLOM_ENTITAS if hasil.get(kunci) not in (TANDA_KOSONG, None, "")),
                "status_lengkap": status,
                "catatan_kosong": catatan or "semua entitas yang berlaku terisi"}
    except Exception as salah:
        kosong = {kunci: TANDA_KOSONG for kunci in KOLOM_ENTITAS}
        return {**kosong, "_ok": False, "_error": str(salah), "file_txt": lintasan_txt.name,
                "jenis_dokumen": "tidak-diketahui", "entitas_berlaku": 0,
                "total_entitas_terisi": 0, "status_lengkap": "gagal",
                "catatan_kosong": f"ERROR: {salah}"}


def tulis_keluaran_txt(lintasan, hasil):
    """Tulis satu entitas per baris memakai label yang mudah dibaca."""
    with open(lintasan, "w", encoding="utf-8") as berkas:
        berkas.write("=== ENTITAS TEREKSTRAKSI ===\n")
        berkas.write(f"File Sumber   : {hasil['file_txt']}\n")
        berkas.write(f"Jenis Dokumen : {hasil['jenis_dokumen']}\n")
        berkas.write(f"Status        : {hasil['status_lengkap']} ({hasil['total_entitas_terisi']}/{hasil['entitas_berlaku']} berlaku)\n")
        berkas.write("-" * 60 + "\n")
        for kunci in KOLOM_ENTITAS:
            berkas.write(f"{LABEL_CETAK[kunci]:17s} : {hasil[kunci]}\n")
        berkas.write("-" * 60 + "\n")
        berkas.write("Catatan Kosong:\n")
        berkas.write(hasil["catatan_kosong"] + "\n")


def demo_satu_file():
    """Demo 1 file: cetak proses scraping, hasil tiap entitas beserta aturan yang dipakai."""
    OUTPUT_CLEAN.mkdir(exist_ok=True)
    contoh = INPUT_TXT / CONTOH_DEMO
    if not contoh.exists():
        kandidat = sorted([x for x in INPUT_TXT.glob("T-*.txt") if "_Pid.B_" in x.name or "_Pid.Sus" in x.name])
        contoh = kandidat[0]
    print(">> Proses scraping entitas")
    print(f">> Membaca: {contoh.name}")
    hasil = ekstrak(contoh)
    print(f">> Jenis dokumen: {hasil['jenis_dokumen']} | Terisi: {hasil['total_entitas_terisi']}/{hasil['entitas_berlaku']} | Status: {hasil['status_lengkap']}")
    print(">> Hasil tiap entitas dan aturan yang dipakai:")
    for kunci in KOLOM_ENTITAS:
        tanda = "✓" if hasil[kunci] != TANDA_KOSONG else "✗"
        print(f"  {tanda} {LABEL_CETAK[kunci]:17s} = {hasil[kunci]}")
        print(f"    aturan: {JEJAK_ATURAN[kunci]}")
    print(">> Alasan nilai kosong:")
    print(f"    {hasil['catatan_kosong']}")
    tulis_keluaran_txt(OUTPUT_CLEAN / f"O-{contoh.name}", hasil)
    print(f">> Ditulis: OUTPUT_CLEAN/O-{contoh.name}")
    return hasil


def proses_batch():
    """Proses semua T-*.txt di CLEAN_TXT/ menjadi O-*.txt + satu CSV rangkuman."""
    OUTPUT_CLEAN.mkdir(exist_ok=True)
    semua = sorted(INPUT_TXT.glob("T-*.txt"))
    print(f">> Memproses {len(semua)} file dari CLEAN_TXT/ ...")
    baris_csv = []
    berhasil = 0
    gagal = 0
    for berkas in semua:
        hasil = ekstrak(berkas)
        if hasil.get("_ok"):
            berhasil += 1
        else:
            gagal += 1
        tulis_keluaran_txt(OUTPUT_CLEAN / f"O-{berkas.name}", hasil)
        baris_csv.append([str(hasil.get(kunci, TANDA_KOSONG) if hasil.get(kunci, TANDA_KOSONG) is not None else TANDA_KOSONG)
                          for kunci in KOLOM_CSV])
    jalur_csv = OUTPUT_CLEAN / "entitas_258.csv"
    with open(jalur_csv, "w", newline="", encoding="utf-8") as csv_file:
        tulis = csv.writer(csv_file)
        tulis.writerow(KOLOM_CSV)
        tulis.writerows(baris_csv)
    print(">> Rekap batch:")
    print(f"  Total file : {len(semua)}")
    print(f"  Berhasil   : {berhasil}")
    print(f"  Gagal      : {gagal}")
    print(f">> CSV       : {jalur_csv}")
    print(f">> TXT       : {OUTPUT_CLEAN} (O-*.txt)")
    return jalur_csv


def tampilkan_ringkasan():
    """Tampilkan 5 data teratas, rekap status/jenis, dan cakupan tiap entitas dari CSV."""
    jalur_csv = OUTPUT_CLEAN / "entitas_258.csv"
    if not jalur_csv.exists():
        print("CSV belum ada. Jalankan cell batch terlebih dahulu.")
        return
    with open(jalur_csv, encoding="utf-8") as csv_file:
        semua_baris = list(csv.DictReader(csv_file))

    print(f">> Menampilkan {BATAS_PREVIEW} data teratas dari {jalur_csv}")
    print("=" * 100)
    for nomor, baris in enumerate(semua_baris[:BATAS_PREVIEW], 1):
        print(f"\n[{nomor}] {baris['file_txt']} | {baris['jenis_dokumen']} | {baris['total_entitas_terisi']}/{baris['entitas_berlaku']} | {baris['status_lengkap']}")
        print("-" * 100)
        for kunci in KOLOM_ENTITAS:
            print(f"  {LABEL_CETAK[kunci]:17s} : {baris[kunci]}")
        print(f"  Catatan     : {baris['catatan_kosong']}")
    print("\n" + "=" * 100)
    print(">> Rekap status:", dict(Counter(x["status_lengkap"] for x in semua_baris)))
    print(">> Rekap jenis :", dict(Counter(x["jenis_dokumen"] for x in semua_baris)))

    print("\n>> Cakupan entitas (terisi dibagi jumlah berlaku)")
    print("-" * 60)
    for kunci in KOLOM_ENTITAS:
        berlaku = sum(1 for x in semua_baris if kunci in himpunan_berlaku(x["jenis_dokumen"]))
        isi = sum(1 for x in semua_baris if x.get(kunci) not in (TANDA_KOSONG, None, ""))
        print(f"  {kunci:20s} : {isi:3d} / {berlaku:3d}")
    print("-" * 60)

### Jalankan tahap 2

In [11]:
# 1) Demo 1 file + jejak aturan
hasil_demo = demo_satu_file()

>> Proses scraping entitas
>> Membaca: T-1016_Pid.B_2026_PN Sby.txt
>> Jenis dokumen: pidana | Terisi: 12/15 | Status: sebagian
>> Hasil tiap entitas dan aturan yang dipakai:
  ✓ Nomor Putusan     = 1016/Pid.B/2026/PN Sby
    aturan: baris judul 'Nomor :' + POLA_NOMOR
  ✓ Nama Terdakwa 1   = Vicky Dubes Borty Bin M Irfan Wijaya
    aturan: label Nama di blok identitas pidana
  ✗ Nama Terdakwa 2   = -
    aturan: label Nama kedua di blok identitas pidana
  ✗ Nama Terdakwa 3   = -
    aturan: label Nama ketiga di blok identitas pidana
  ✓ Tuntutan Pidana   = yang secara melawan hukum memiliki suatu barang yang sebagian atau seluruhnya milik orang lain, yang ada dalam kekuasaannya bukan
    aturan: amar 1 Tuntutan + frasa sesudah 'bersalah melakukan'
  ✗ Tuntutan KUHP     = -
    aturan: amar 1 Tuntutan + pola 'Pasal ... KUHP'
  ✓ Tuntutan Hukuman  = 1 (satu) tahun 3 (tiga) bulan
    aturan: amar 2 Tuntutan + durasi sesudah 'selama'
  ✓ Putusan Pidana    = Penggelepan
    aturan: amar 1 M

In [12]:
# 2) Batch semua file -> O-*.txt + entitas_258.csv
jalur_csv = proses_batch()

>> Memproses 258 file dari CLEAN_TXT/ ...
>> Rekap batch:
  Total file : 258
  Berhasil   : 258
  Gagal      : 0
>> CSV       : OUTPUT_CLEAN/entitas_258.csv
>> TXT       : OUTPUT_CLEAN (O-*.txt)


In [13]:
# 3) 5 data teratas, rekap status/jenis, dan cakupan tiap entitas
tampilkan_ringkasan()

>> Menampilkan 5 data teratas dari OUTPUT_CLEAN/entitas_258.csv

[1] T-1002_Pid.Sus-LH_2026_PN Sby.txt | pidana | 12/15 | sebagian
----------------------------------------------------------------------------------------------------
  Nomor Putusan     : 1002/Pid.Sus-LH/2026/PN Sby
  Nama Terdakwa 1   : HENDRIK SETIAWAN BIN (alm) ABDUL HALIM
  Nama Terdakwa 2   : -
  Nama Terdakwa 3   : -
  Tuntutan Pidana   : menyalagunakan Pengangkutan dan/atau Niaga Bahan Bakar Minyak, Bahan Bakar Gas, dan/atau liquefied petroleum gas yang disbusidi dan/atau penyediaan dan pendistribusiannya diberikan penugasan pemerintah
  Tuntutan KUHP     : -
  Tuntutan Hukuman  : 1 (satu) Tahun dan 6 (enam) bulan
  Putusan Pidana    : menyalahgunakan niaga bahan bakar minyak yang disubsidi dan atau penyediaan dan pendistribusiannya diberikan penugasan pemerintah
  Putusan Hukuman   : 1 (satu) tahun dan 2 (dua) bulan
  Tanggal Putusan   : 31 Agustus 2026
  Hakim Ketua       : Alex Adam Faisal, S.H., M.H.
  Hakim A

In [17]:
# ===== Gabungkan 5 hasil O-*.txt menjadi satu PDF (IE-RuleBased-Hasil-<nim>.pdf) =====
import textwrap
from pathlib import Path
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Preformatted, PageBreak, Spacer

NIM = "230411100145"                       # <-- ganti dengan NIM Anda
FILE_PILIHAN = [                       # <-- ganti dengan 5 file pilihan Anda
    "O-T-1016_Pid.B_2026_PN Sby.txt",
    "O-T-191_Pid.B_2018_PN SBY.txt",
    "O-T-1184_Pid.Sus_2026_PN Sby.txt",
    "O-T-55_Pid.Sus-TPK_2026_PN Sby.txt",
    "O-T-1057_Pid.B_2026_PN Sby.txt",
]
LEBAR_BARIS = 88                       # karakter per baris (Courier 8 pt di A4)

gaya = getSampleStyleSheet()
gaya_kode = gaya["Code"].clone("kode", fontName="Courier", fontSize=8, leading=10)
isi = []
for nomor, nama in enumerate(FILE_PILIHAN, 1):
    teks = (Path("OUTPUT_CLEAN") / nama).read_text(encoding="utf-8")
    baris = []
    for b in teks.splitlines():
        baris += textwrap.wrap(b, LEBAR_BARIS, subsequent_indent="    ") or [""]
    isi.append(Paragraph(f"Hasil {nomor}: {nama}", gaya["Heading3"]))
    isi.append(Spacer(1, 0.3 * cm))
    isi.append(Preformatted("\n".join(baris), gaya_kode))
    if nomor < len(FILE_PILIHAN):
        isi.append(PageBreak())

SimpleDocTemplate(f"IE-RuleBased-Hasil-{NIM}.pdf", pagesize=A4,
                  leftMargin=2*cm, rightMargin=2*cm, topMargin=2*cm, bottomMargin=2*cm).build(isi)
print(f"Selesai: IE-RuleBased-Hasil-{NIM}.pdf ({len(FILE_PILIHAN)} hasil)")

Selesai: IE-RuleBased-Hasil-230411100145.pdf (5 hasil)
